# 04 — Macro Merge (Eurostat + OECD EPL + Welfare Regime)

Build the L2 macro panel and L3 institutional moderators, then merge them onto the individual panel.

## What this notebook does

1. Maps each ESS round to a fielding year for the macro merge (R6→2012, R7→2014, R8→2016, R9→2018, R10→2021, R11→2023).
2. Fetches Eurostat:
   - `nama_10_gdp` (GDP growth, % change on prev. year, chain-linked volume)
   - `une_rt_a` (annual unemployment rate, ages 15–74, both sexes, % active)
   - `prc_hicp_aind` (annual HICP rate of change, % — i.e. inflation)
3. Loads L3 institutions: OECD EPL_v1 (2007 baseline) and welfare-regime typology.
4. Joins all of them onto the individual panel from notebook 03 to produce `data/analysis/analysis.parquet`.
5. Reports coverage and country-year counts; re-checks the Mundlak identity after the joins.

## Coverage caveats

* **IL and RU** are not in Eurostat → their ESS rows have NaN macro covariates. The 13,447 individuals (4.9% of panel) drop out of M2+ models via listwise deletion.
* **Non-OECD ESS countries** (most Western Balkans, post-Soviet) have NaN `epl_c`. M6 (institutional moderation) excludes them; M0–M5 unaffected.

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().resolve()
while REPO_ROOT.name != "MLA" and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.mla.macro import build_macro_panel  # noqa: E402
from src.mla.institutions import build_l3_frame  # noqa: E402

INTERIM_DIR = REPO_ROOT / "data" / "interim"
ANALYSIS_DIR = REPO_ROOT / "data" / "analysis"
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
REPO_ROOT

PosixPath('/Users/karlalucic/Code/coursework/KUL/2sem/MLA')

## 1. Load the panel-with-L2-exposure from notebook 03

In [2]:
panel = pd.read_parquet(INTERIM_DIR / "ess_panel_with_l2.parquet")
print(f"shape: {panel.shape}")
geos = sorted(panel["cntry"].unique().tolist())
print(f"unique cntry codes ({len(geos)}): {geos}")

shape: (276491, 30)
unique cntry codes (36): ['AL', 'AT', 'BE', 'BG', 'CH', 'CY', 'CZ', 'DE', 'DK', 'EE', 'ES', 'FI', 'FR', 'GB', 'GR', 'HR', 'HU', 'IE', 'IL', 'IS', 'IT', 'LT', 'LV', 'ME', 'MK', 'NL', 'NO', 'PL', 'PT', 'RS', 'RU', 'SE', 'SI', 'SK', 'UA', 'XK']


## 2. Round → fielding year mapping

In [3]:
ROUND_YEAR = {6: 2012, 7: 2014, 8: 2016, 9: 2018, 10: 2021, 11: 2023}
panel["year"] = panel["essround"].map(ROUND_YEAR).astype("Int16")
panel[["essround", "year"]].drop_duplicates().sort_values("essround")

,essround,year
0,6,2012
54673,7,2014
94858,8,2016
139245,9,2018
188764,10,2021
226375,11,2023


## 3. Fetch Eurostat macro panel

Cached on disk under `data/raw/eurostat/`; subsequent runs read from cache.

In [4]:
macro = build_macro_panel(geos=geos, year_min=2010, year_max=2024)
print(f"macro shape: {macro.shape}")
print()
print("coverage by indicator (non-null):")
print(macro[["gdp_growth", "unemp_rate", "hicp_inflation"]].notna().sum())
print()
missing_geos = sorted(set(geos) - set(macro["cntry"].unique()))
print(f"ESS countries not in Eurostat (NaN macro): {missing_geos}")

macro shape: (504, 5)

coverage by indicator (non-null):
gdp_growth        504
unemp_rate        445
hicp_inflation    470
dtype: int64

ESS countries not in Eurostat (NaN macro): ['IL', 'RU']


In [5]:
macro.head(8)

,cntry,year,gdp_growth,unemp_rate,hicp_inflation
0,AL,2010,3.0,NaN,NaN
1,AL,2011,2.5,NaN,NaN
2,AL,2012,1.0,NaN,NaN
3,AL,2013,1.7,NaN,NaN
4,AL,2014,2.2,NaN,NaN
5,AL,2015,2.2,NaN,NaN
6,AL,2016,3.9,NaN,NaN
7,AL,2017,3.3,NaN,3.2


## 4. L3 institutions: EPL_v1 (2007) + welfare regime

In [6]:
l3 = build_l3_frame()
print(f"L3 frame: {l3.shape}")
print()
print("welfare regime distribution (number of countries per class):")
print(l3["welfare_regime"].value_counts(dropna=False))
print()
print("EPL coverage:")
print(f"  countries with non-null epl_c: {l3['epl_c'].notna().sum()}")
print(f"  ESS countries missing EPL: {sorted(set(geos) - set(l3.dropna(subset=['epl_c'])['cntry']))}")
l3

L3 frame: (34, 3)

welfare regime distribution (number of countries per class):
welfare_regime
eastern-european            16
conservative-corporatist     6
mediterranean                5
social-democratic            5
liberal                      2
Name: count, dtype: int64

EPL coverage:
  countries with non-null epl_c: 23
  ESS countries missing EPL: ['AL', 'BG', 'CY', 'HR', 'IL', 'LT', 'LV', 'ME', 'MK', 'RS', 'RU', 'UA', 'XK']


,cntry,epl_c,welfare_regime
0,AL,NaN,eastern-european
1,AT,2.37,conservative-corporatist
2,BE,1.73,conservative-corporatist
3,BG,NaN,eastern-european
4,CH,1.60,conservative-corporatist
5,CY,NaN,mediterranean
6,CZ,3.05,eastern-european
7,DE,2.85,conservative-corporatist
8,DK,1.63,social-democratic
9,EE,1.95,eastern-european


## 5. Build the analysis frame

In [7]:
analysis = (
    panel.merge(macro, on=["cntry", "year"], how="left", validate="many_to_one")
    .merge(l3, on="cntry", how="left", validate="many_to_one")
)
print(f"analysis frame shape: {analysis.shape}")
assert len(analysis) == len(panel), "merge changed panel size"
print()
print("new columns:", [c for c in analysis.columns if c not in panel.columns])

analysis frame shape: (276491, 36)

new columns: ['gdp_growth', 'unemp_rate', 'hicp_inflation', 'epl_c', 'welfare_regime']


### Coverage of analysis-ready rows

An individual is **analysis-ready** for the M2+ models if all the following are present:
* trust composite items (`trstprl`, `trstlgl`, `stfdem`)
* `genai_i` (ILO–NASK score for their occupation)
* macro: `gdp_growth`, `unemp_rate`, `hicp_inflation`
* essential L1 controls: `eisced`, `agea`, `gndr`

In [8]:
TRUST_COLS = ["trstprl", "trstlgl", "stfdem"]
MACRO_COLS = ["gdp_growth", "unemp_rate", "hicp_inflation"]
L1_REQUIRED = ["genai_i", "eisced", "agea", "gndr"]

# ESS encodes missing as sentinel codes (66/77/88/99 etc.) for ordinal items.
# Treat trust > 10 as missing for the trust composite.
trust_valid = pd.concat(
    [analysis[c].between(0, 10) for c in TRUST_COLS], axis=1
).all(axis=1)
ready = (
    trust_valid
    & analysis[MACRO_COLS].notna().all(axis=1)
    & analysis[L1_REQUIRED].notna().all(axis=1)
)
print(f"analysis-ready rows: {ready.sum():,} of {len(analysis):,} ({ready.mean():.1%})")
print()
print("analysis-ready coverage by round:")
print(analysis[ready].groupby("essround").size().rename("n"))

analysis-ready rows: 194,227 of 276,491 (70.2%)

analysis-ready coverage by round:
essround
6     34829
7     28989
8     30519
9     38285
10    27144
11    34461
Name: n, dtype: int64


## 6. Persist `data/analysis/analysis.parquet`

In [9]:
out_path = ANALYSIS_DIR / "analysis.parquet"
analysis.to_parquet(out_path, index=False)
print(f"wrote {out_path}  ({out_path.stat().st_size / 1e6:.1f} MB)")

wrote /Users/karlalucic/Code/coursework/KUL/2sem/MLA/data/analysis/analysis.parquet  (6.7 MB)


## 7. Mundlak identity check after merge

Notebook 03 verifies `exposure_ct == exposure_within + exposure_between` element-wise. This cell re-verifies the same identity after the macro and institutional joins to make sure no rounding drift was introduced.

In [10]:
gap = (
    analysis["exposure_ct"]
    - analysis["exposure_ct_between"]
    - analysis["exposure_ct_within"]
).abs().max()
static_gap = (
    analysis["exposure_ct_static"]
    - analysis["exposure_ct_static_between"]
    - analysis["exposure_ct_static_within"]
).abs().max()
print(f"max identity gap (vintage): {gap:.2e}")
print(f"max identity gap (static):  {static_gap:.2e}")
print(f"Mundlak identity after merge: {'PASS' if max(gap, static_gap) < 1e-12 else 'FAIL'}")

max identity gap (vintage): 0.00e+00
max identity gap (static):  0.00e+00
Mundlak identity after merge: PASS
